### PDF Document Processing Pipeline
**Objective:** Extract text from product Documentation PDF files and store it in unity Catalog table
**Source Location: ** /Volumes/agentic_catalog/agentic_schema/customer_service/product_docs/
**Target Table:** agentic_catalog.agentic_schema.products_docs

**Schema:**

- Product_name - PDF file name without extension
- product_doc - Extracted text content

 

In [0]:
# Install pypdf Library before importing using command %pip install pypdf
import pypdf


### Step 1: Discover PDF files

In [0]:

# Define the source path
source_path = '/Volumes/agentic_catalog/agentic_schema/customer_service/product_docs/'

#List all files in directory
files = dbutils.fs.ls(source_path)

#Filter for PDF files only (list comprehension)
pdf_files = [f for f in files if f.name.endswith('.pdf')]

print(f"found {len(pdf_files)} PDF Files:")

for pdf in pdf_files:
    print(f" - {pdf.name}")
 

### Step 2: Extract Text from PDF

We'll define a function to:
    1. Read each PDF file using pypdf
    2. Extract Text from all pages
    3. Return the product name and extracted text


In [0]:
def extract_text_from_pdf(file_path):
    """
    Etract Text from pdf

    Args:
        file_path: Full path to the pdf file.

    Returns:
        Extracted text as a string
    """

    try:
        with open(file_path,'rb') as f:
            pdf_reader = pypdf.PdfReader(f)

            text_content = ""
            for page in pdf_reader.pages:
                text_content += page.extract_text()

            return text_content
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return None
    
print("Text Extraction function defined successfully!")


### Step 3: Porcess All PDF file


In [0]:
#Initialize list to store results
product_data = []

for pdf_file in pdf_files:
    print(f"Processing: {pdf_file.name}")

    #Extract product name (remove) .pdf extension
    product_name = pdf_file.name.replace('.pdf','')

    #Extract text from PDF
    full_path = f"{source_path}{pdf_file.name}"
    product_doc = extract_text_from_pdf(full_path)

    if product_doc:
        product_data.append(
            {
                'product_name': product_name,
                'product_doc' : product_doc
            }
        )
        print(f" Successfully extracted {len(product_doc)} characters")

    else:
        print(f"Failed to extract text")

print(f"\nTotal documents processed: {len(product_data)}")

### Step 4: Create Delta Table 

We'll:
- Convert the etxracted data into a spark Dataframe
- Write it to the unity catalog table agentic_catalog.agentic_schema.product_docs
- Use overwrite mode to replace any existing data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

#Define schema for DataFrame
schema = StructType(
    [
        StructField('product_name', StringType(), False),
        StructField('product_doc', StringType(), True)
    ]
)

#Create Spark DataFrame from the extracted data
df = spark.createDataFrame(product_data, schema=schema)

#Display dataframe schema & count
print(f"DataFrame created with {df.count()} rows\n")
df.printSchema()

# write to unity Catalog table
table_name = "agentic_catalog.agentic_schema.product_docs"
print(f"\n Writing data to table: {table_name}")

df.write\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .saveAsTable(table_name)

print(f"Successfully created table: {table_name}")

### Step 5: Verify the Table

In [0]:
%sql
select * from agentic_catalog.agentic_schema.product_docs

### Creating Delta Tables from CSV source files

In [0]:
# Volume path
volume_path = '/Volumes/agentic_catalog/agentic_schema/customer_service'
file_path = dbutils.fs.ls(volume_path)

# filtering only csv files within volume
csv_files = [f for f in file_path if f.name.endswith('.csv')]
print(f"Found {len(csv_files)} csv files for processing")

# function to create delta table
def create_delta_table(csv_file_path):
    df = spark.read\
        .format('csv')\
        .option('header', 'true')\
        .option('inferSchema', 'true')\
        .load(f"{csv_file_path.path}")
    df.printSchema()

    table_name = f"agentic_catalog.agentic_schema.{csv_file_path.name.replace('.csv','')}"
    print(f"Writing data to table: {table_name}")

    df.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema","true")\
        .saveAsTable(table_name)

    print(f"Successfully created table: {table_name}")

# loop through all csv files and create delta table
for csv_file in csv_files:
    create_delta_table(csv_file)


### Verifying Tables generated from csv files

In [0]:
%sql
select * from agentic_catalog.agentic_schema.products